# Teacher: label the biographies on Colab

Step 3B of the guide: an open-source teacher on Colab's T4 labels `raw.jsonl`, then the gold test set
and the train/val/test split are built (step 4). Run it with **Runtime → Change runtime type → T4 GPU**.

**Credentials:** `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY` (the `pt-bio-extract-colab` user) and `DATA_BUCKET`.
In the Colab website, cell 1 reads them from **Secrets** (key icon, on the left). From VS Code or Antigravity,
Secrets can't be read, so cell 1 asks for each one in a password box.

**If Colab disconnects**, run cell 1 again, then the labeling cell: it resumes where it stopped,
because `labels.jsonl` lives on Google Drive.

**Same teacher, same prompt, same labels:** generation is greedy, so labeling again with the settings
already used for `v1` gives back the same labels. Change `TEACHER` (or the prompt in `common.py`)
and set `VERSION = "v2"`, so the `v1` results stay comparable.

## 1. Set up

In [ ]:
TEACHER = "Qwen/Qwen2.5-7B-Instruct"  # bigger and slower, but better: "Qwen/Qwen2.5-14B-Instruct"
VERSION = "v1"                        # use "v2" for a new teacher or prompt: v1 stays as it is

!pip install -q -U transformers bitsandbytes accelerate pydantic awscli

import json, os
from google.colab import drive, userdata

from getpass import getpass

def secret(name):
    """Colab Secrets in the browser; outside it (VS Code, Antigravity) they can't be read,
    so ask for the value instead. It stays in this session's memory only."""
    if os.environ.get(name):
        return os.environ[name]
    try:
        return userdata.get(name).strip()  # a stray space in a secret breaks the AWS credentials
    except Exception:  # TimeoutException outside the browser, or a missing secret
        return getpass(f"{name}: ").strip()

for k in ("AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "DATA_BUCKET"):
    os.environ[k] = secret(k)
os.environ["AWS_DEFAULT_REGION"] = "eu-west-1"
# Also a Python variable: shell lines below use {DATA_BUCKET}, never $DATA_BUCKET mixed with {...}
DATA_BUCKET = os.environ["DATA_BUCKET"]
S3 = f"s3://{DATA_BUCKET}/{VERSION}"

drive.mount("/content/drive")  # anything slow goes to Drive: it survives disconnects
DRIVE = f"/content/drive/MyDrive/wiki-bio-distill/{VERSION}"
os.makedirs(DRIVE, exist_ok=True)

# The repository is public: no token needed
url = "https://github.com/pinthoz/wiki-bio-distill.git"
if not os.path.exists("/content/repo"):
    !git clone -q {url} /content/repo
%cd /content/repo
!git pull -q
!mkdir -p data

!aws sts get-caller-identity --query Arn --output text   # must end in user/pt-bio-extract-colab
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Fetch the texts and link the output file to Drive

In [ ]:
# raw.jsonl is the same for every version: the texts do not change, only the labels
!aws s3 cp s3://{DATA_BUCKET}/v1/raw.jsonl data/ --only-show-errors
!wc -l data/raw.jsonl

# labels.jsonl lives on Drive; data/labels.jsonl is just a link to it
!touch {DRIVE}/labels.jsonl
!ln -sf {DRIVE}/labels.jsonl data/labels.jsonl
!wc -l data/labels.jsonl   # more than 0 lines: an earlier session already labeled these

## 3. Trial run with 50 examples

Check the error rate, the speed and a few labels before launching the full run.
If the labels look wrong, fix the prompt in `common.py`, push it, run cell 1 again,
empty the labels with `!rm {DRIVE}/labels.jsonl` and run cell 2 and this one again.

In [ ]:
!python label_local.py --model {TEACHER} --limit 50

rows = {r["id"]: r for r in map(json.loads, open("data/raw.jsonl", encoding="utf-8"))}
labels = [json.loads(l) for l in open("data/labels.jsonl", encoding="utf-8")]
print(f"{len(labels)} labeled, {sum('error' in l for l in labels)} with errors\n")
for l in labels[:5]:
    print(rows[l["id"]]["text"][:200], "…")
    print(json.dumps(l.get("label") or l, ensure_ascii=False, indent=2), "\n")

## 4. Label the rest

For 3000 biographies, expect one to two hours with the 7B. The script shows the time left.

In [ ]:
!python label_local.py --model {TEACHER}

labels = [json.loads(l) for l in open("data/labels.jsonl", encoding="utf-8")]
errors = [l for l in labels if "error" in l]
print(f"{len(labels)} labeled, {len(errors)} with errors ({len(errors) / len(labels):.1%})")
for e in errors[:5]:
    print(e["id"], "|", e.get("raw", "")[:150])
!aws s3 cp data/labels.jsonl {S3}/labels.jsonl --only-show-errors

## 5. Gold test set and split (step 4)

`--accept-all` takes the teacher's label as gold for the 150 test examples, without review.
That makes the test measure agreement with the teacher, not correctness: for the final numbers,
review them by hand on your laptop instead (`python review.py --n 150`), then run `split.py` there.

In [ ]:
!rm -f data/gold.jsonl
!python review.py --n 150 --accept-all
!python split.py

for f in ("gold", "train", "val", "test"):
    !aws s3 cp data/{f}.jsonl {S3}/{f}.jsonl --only-show-errors
    !cp data/{f}.jsonl {DRIVE}/
!aws s3 ls {S3}/

Done. Next: `02_train_student.ipynb` (the Qwen student) and `03_bert_student.ipynb` (the BERT student), with the same `VERSION`.